# 04. Climate (GDD) stratification of training data

## Setup

In [ ]:
from pathlib import Path
import json
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").is_file():
    if root == root.parent:
        raise RuntimeError("Open this notebook inside the project repository")
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # experiments package

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from experiments.pilots import climate, clustering, data, earth_engine, evaluation, features
from experiments.tuning import tuning
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
AREA = "pilots"


def show_and_save(figure, run, name):
    figure.tight_layout()
    figure.savefig(run / f"{name}.png", dpi=150)
    display(figure)
    plt.close(figure)


def repeated(name, X, rows, ok, seeds=(1, 2, 3), params=evaluation.PARAMS):
    """Mean and SD over random-forest seeds of every evaluation view."""
    summaries, details = [], []
    for seed in seeds:
        summary, holdout = evaluation.evaluate(name, X, rows, ok, params, seed)
        summaries.append({**summary, "seed": seed})
        details.extend({**r, "seed": seed} for r in holdout)
    return summaries, details


def summarize(summaries):
    table = pd.DataFrame(summaries)
    metrics = ["random_split_macro_f1", "spatial_cv_macro_f1", "dataset_holdout_macro_f1", "jecam_macro_f1",
               "jecam_carrot_f1", "jecam_cabbage_f1", "jecam_potato_f1"]
    grouped = table.groupby("feature_set", sort=False)
    out = grouped[metrics].agg(["mean", "std"]).round(3)
    out.insert(0, ("pixels_used", ""), grouped["pixels_used"].first())
    return out

## Inputs

In [ ]:
subset_run = ""  # Optional pipeline_outputs/pilots/subset_<id>; blank uses the latest
subset_run = Path(subset_run) if subset_run else tuning.latest_run(root, "subset", area=AREA)
rows, arrays = data.load(subset_run)
cube = features.stack(arrays["s2_values"])
valid = arrays["s2_valid"]
print("Subset:", subset_run, "| pixels:", len(rows))

In [ ]:
era5 = json.loads((subset_run / "era5.json").read_text())
paoay = json.loads((subset_run / "paoay.json").read_text())
descriptor = {g: climate.descriptors(v) for g, v in era5.items() if v is not None and None not in v}
paoay_descriptor = climate.descriptors(paoay["era5"])
X, ok, columns = features.harmonic_features(cube, valid)
X = np.nan_to_num(X)
ok = ok & np.asarray([r["partition_group"] in descriptor for r in rows])
print("Pixels with features and temperature:", int(ok.sum()))

## Climate descriptors by source and target

In [ ]:
parcels = {}
for r in rows:
    if r["partition_group"] in descriptor:
        parcels[r["partition_group"]] = {"source": r["source_dataset"], "partition": r["partition"], **descriptor[r["partition_group"]]}
table = pd.DataFrame(parcels.values())
summary = table.groupby("source")[list(climate.DESCRIPTORS)].median()
summary.loc["paoay (ERA5)"] = [paoay_descriptor[d] for d in climate.DESCRIPTORS]
display(summary.round(1))

## Distance of each development source to Paoay and to JECAM

In [ ]:
development = [g for g, p in parcels.items() if p["partition"] in ("train", "validation")]
jecam = [descriptor[g] for g, p in parcels.items() if p["partition"] == "test"]
to_paoay = dict(zip(development, climate.distances([descriptor[g] for g in development], [paoay_descriptor])))
to_jecam = dict(zip(development, climate.distances([descriptor[g] for g in development], jecam)))
display(pd.DataFrame([{"source": parcels[g]["source"], "to_paoay": to_paoay[g], "to_jecam": to_jecam[g]} for g in development])
        .groupby("source").median().round(2))

## Climate-matched training evaluated on JECAM and dataset holdout

In [ ]:
from experiments.transfer import transfer

def matched(strategy):
    def weights_for(train_rows, target_rows):
        targets = [descriptor[r["partition_group"]] for r in target_rows]
        groups = sorted({r["partition_group"] for r in train_rows})
        distance = dict(zip(groups, climate.distances([descriptor[g] for g in groups], targets)))
        base = transfer.training_weights(train_rows)
        if strategy == "nearest_half":
            return climate.nearest_share_weights(train_rows, distance, base, keep=.5)
        return climate.similarity_weights(train_rows, distance, base, bandwidth=1.0)
    return weights_for

run = tuning.new_run(root, "climate", subset_run / "meta.json", area=AREA)
records, details = [], []
for strategy in ("all_sources", "nearest_half", "soft_similarity"):
    weights_for = None if strategy == "all_sources" else matched(strategy)
    for seed in (1, 2, 3):
        test = evaluation.external_test(X, rows, ok, seed=seed, weights_for=weights_for)
        holdout = evaluation.dataset_holdout(X, rows, ok, seed=seed, weights_for=weights_for)
        scored = [h for h in holdout if h["status"] == "scored"]
        records.append({"strategy": strategy, "seed": seed, "jecam_macro_f1": test["macro_f1"],
                        **{f"jecam_{c}_f1": test["per_crop"][c]["f1"] for c in evaluation.CROPS},
                        "dataset_holdout_macro_f1": float(np.mean([h["macro_f1_scored_crops"] for h in scored]))})
        details += [{"strategy": strategy, "seed": seed, **h} for h in holdout]
    print(strategy, "done", flush=True)
tuning.save_records(run, "summary", records)
tuning.save_records(run, "dataset_holdout", details)
display(pd.DataFrame(records).groupby("strategy", sort=False).agg(["mean", "std"]).drop(columns="seed").round(3))

## Figure

In [ ]:
table = pd.DataFrame(records).groupby("strategy", sort=False)[["jecam_macro_f1", "dataset_holdout_macro_f1"]].mean()
figure, axis = plt.subplots(figsize=(8, 4))
table.plot(kind="bar", ax=axis)
axis.set_ylim(0, 1)
axis.set_ylabel("Parcel macro-F1 (mean of seeds)")
show_and_save(figure, run, "climate_stratification")
print("Run folder:", run)